# PS S6E6 - 049 GPU Logistic Regression Stacker add048 using own OOF/pred assets only

Experiment: `exp_20260614_049_gpu_logreg_stacker_add048_own`

Purpose:
- Run a GPU-accelerated multinomial logistic regression stacker.
- Use only our own OOF/pred probability assets.
- Start from the 045 GPU LogReg add044 base asset pool.
- Add `048 OVR XGB seed2027 FE/fold variant` as the new XGB-family auxiliary material.
- Keep `044 shared001 updated seed2027/fold variant` because this notebook is based on 045.
- Deliberately do not add `047 multiclass XGB` because it underperformed 030/048 and had weak STAR recall.
- Deliberately do not add `042 RealMLP v5 seed2026/fold variant` because 043 add042 underperformed 040 on both CV and Public LB.
- Do not use external submission files or other Kaggler predictions.
- Generate OOF/pred `.npy`, submissions, diagnostics, `oof_registry.csv`, and `memo.yml`.

Key reference:
- 040 GPU LogReg add039: CV `0.9702017877238539`, Public LB `0.97069`.
- 045 GPU LogReg add044: CV `0.9702445493383917`, Public LB `0.97040`.
- 046 blend add042/044/045: CV `0.9702533348410616`, Public LB `0.97042`.
- 048 OVR XGB seed2027 FE/fold variant: CV `0.9636973807714805`, Public LB `0.96444`.

Decision:
- 049 tests the marginal stacker value of 048.
- 047 is not included.
- 042 is not included.


In [1]:

# ============================================================
# 0. Imports / Config
# ============================================================

import gc
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260614_049_gpu_logreg_stacker_add048_own"
CREATED_AT = "2026-06-14"
SEED = 42

TARGET_COL = "class"
ID_COL = "id"
CLASS_NAMES = ["GALAXY", "QSO", "STAR"]
TARGET_MAP = {"GALAXY": 0, "QSO": 1, "STAR": 2}
INV_MAP = {v: k for k, v in TARGET_MAP.items()}

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

INPUT_ROOT = Path("/kaggle/input")
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

# GPU logistic regression config
N_FOLDS = 5
N_SEEDS = 5
STACKER_SEEDS = list(range(42, 42 + N_SEEDS))
EPOCHS = 1000
LR = 0.01
C_REG = 0.1
BOOST_STAR_WEIGHT = 1.0
EPS = 1e-15
LOGIT_CLIP = 30.0

# Probability material handling
USE_LOGIT_INPUT = True
INCLUDE_OPTIONAL_033 = True

# Output paths. Default *_proba.npy points to the best OOF-calibrated probability material
# among raw stacker proba and bias-tuned stacker proba.
OOF_NPY = OUTDIR / f"oof_{EXP_ID}_proba.npy"
PRED_NPY = OUTDIR / f"pred_{EXP_ID}_proba.npy"
OOF_RAW_NPY = OUTDIR / f"oof_raw_{EXP_ID}_proba.npy"
PRED_RAW_NPY = OUTDIR / f"pred_raw_{EXP_ID}_proba.npy"
OOF_TUNED_NPY = OUTDIR / f"oof_tuned_{EXP_ID}_proba.npy"
PRED_TUNED_NPY = OUTDIR / f"pred_tuned_{EXP_ID}_proba.npy"

SUBMISSION_PATH = OUTDIR / f"submission_{EXP_ID}.csv"
SUBMISSION_RAW_PATH = OUTDIR / f"submission_raw_{EXP_ID}.csv"
SUBMISSION_TUNED_PATH = OUTDIR / f"submission_tuned_{EXP_ID}.csv"

CV_RESULT_PATH = OUTDIR / f"cv_result_{EXP_ID}.json"
SUMMARY_PATH = OUTDIR / "gpu_logreg_stacker_summary.json"
FOLD_SCORES_PATH = OUTDIR / f"fold_scores_{EXP_ID}.csv"
SEED_SCORES_PATH = OUTDIR / f"seed_scores_{EXP_ID}.csv"
BASE_SCORES_PATH = OUTDIR / f"base_model_scores_{EXP_ID}.csv"
LOADED_SPECS_PATH = OUTDIR / f"loaded_model_specs_{EXP_ID}.csv"
FEATURE_IMPORTANCE_PATH = OUTDIR / f"stacker_importance_{EXP_ID}.csv"
CONFUSION_RAW_PATH = OUTDIR / f"confusion_matrix_raw_{EXP_ID}.csv"
CONFUSION_TUNED_PATH = OUTDIR / f"confusion_matrix_tuned_{EXP_ID}.csv"

# Own OOF/pred assets only. No external submission or public notebook predictions.
# 033 is optional in the source notebook, but here it is normally expected to be present.
# 035, 037, 039, 044, and 048 are required for this add048 stacker run.
# 042 is deliberately excluded because 043 add042 underperformed 040 on both CV and Public LB.
# 047 is deliberately excluded because the multiclass XGB variant underperformed 030/048 and had weak STAR recall.
MODEL_SPECS = [
    {
        "key": "015_realmlp_seed42",
        "exp_id": "exp_20260605_015_shared001_realmlp_as_is",
        "family": "RealMLP",
        "role": "stable_single_realmlp_backup",
        "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "cv": 0.9681693449222352,
        "public_lb": 0.96977,
        "optional": False,
    },
    {
        "key": "017_realmlp_bias",
        "exp_id": "exp_20260606_017_bias_search_on_015_realmlp",
        "family": "RealMLP",
        "role": "previous_best_realmlp_bias_backup",
        "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "cv": 0.9683022653955233,
        "public_lb": 0.96985,
        "optional": False,
    },
    {
        "key": "018_xgb_shared003",
        "exp_id": "exp_20260606_018_shared003_xgb_as_is",
        "family": "XGBoost",
        "role": "effective_blend_material",
        "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "cv": 0.965207986418628,
        "public_lb": 0.96578,
        "optional": False,
    },
    {
        "key": "019_blend_best",
        "exp_id": "exp_20260607_019_blend_add018_xgb_check",
        "family": "Blend",
        "role": "previous_public_confirmed_best",
        "oof": "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
        "pred": "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
        "cv": 0.968872315017554,
        "public_lb": 0.97000,
        "optional": False,
    },
    {
        "key": "020_blend_bias",
        "exp_id": "exp_20260607_020_bias_search_on_019_best_blend",
        "family": "Blend",
        "role": "cv_trusted_attack_reference",
        "oof": "oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy",
        "pred": "pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy",
        "cv": 0.9692240443617589,
        "public_lb": 0.96969,
        "optional": False,
    },
    {
        "key": "024_seed_ensemble_blend",
        "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check",
        "family": "Blend",
        "role": "seed_ensemble_blend_reference",
        "oof": "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy",
        "cv": 0.9694803109983217,
        "public_lb": 0.96958,
        "optional": False,
    },
    {
        "key": "026_realmlp_v5",
        "exp_id": "exp_20260608_026_realmlp_v5_as_is",
        "family": "RealMLP",
        "role": "realmlp_v5_single_model_candidate",
        "oof": "oof_exp_20260608_026_realmlp_v5_as_is_proba.npy",
        "pred": "pred_exp_20260608_026_realmlp_v5_as_is_proba.npy",
        "cv": 0.9690389777981231,
        "public_lb": 0.96979,
        "optional": False,
    },
    {
        "key": "027_blend_add026",
        "exp_id": "exp_20260608_027_blend_add026_realmlp_v5_check",
        "family": "Blend",
        "role": "previous_cv_trusted_slot",
        "oof": "oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy",
        "cv": 0.9695425457477898,
        "public_lb": 0.96975,
        "optional": False,
    },
    {
        "key": "028_cat_v3",
        "exp_id": "exp_20260608_028_cat_v3_as_is",
        "family": "CatBoost",
        "role": "catboost_v3_family_aux_material",
        "oof": "oof_exp_20260608_028_cat_v3_as_is_proba.npy",
        "pred": "pred_exp_20260608_028_cat_v3_as_is_proba.npy",
        "cv": 0.9688146470512534,
        "public_lb": 0.96969,
        "optional": False,
    },
    {
        "key": "029_blend_add028",
        "exp_id": "exp_20260608_029_blend_add028_cat_v3_check",
        "family": "Blend",
        "role": "previous_public_confirmed_and_cv_trusted_best",
        "oof": "oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy",
        "cv": 0.9700023026377228,
        "public_lb": 0.970036,
        "optional": False,
    },
    {
        "key": "030_ovr_xgb",
        "exp_id": "exp_20260608_030_ovr_xgb_as_is",
        "family": "XGBoost",
        "role": "weak_xgb_ovr_family_material",
        "oof": "oof_exp_20260608_030_ovr_xgb_as_is_proba.npy",
        "pred": "pred_exp_20260608_030_ovr_xgb_as_is_proba.npy",
        "cv": 0.9609971296999271,
        "public_lb": 0.96118,
        "optional": False,
    },
    {
        "key": "031_blend_add030",
        "exp_id": "exp_20260608_031_blend_add030_ovr_xgb_check",
        "family": "Blend",
        "role": "public_confirmed_current_best_before_034",
        "oof": "oof_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy",
        "cv": 0.9700333620193362,
        "public_lb": 0.97043,
        "optional": False,
    },
    {
        "key": "032_ovr_tabm",
        "exp_id": "exp_20260609_032_ovr_tabm_as_is",
        "family": "TabM",
        "role": "tabm_ovr_family_aux_material",
        "oof": "oof_exp_20260609_032_ovr_tabm_as_is_proba.npy",
        "pred": "pred_exp_20260609_032_ovr_tabm_as_is_proba.npy",
        "cv": 0.9610105651284856,
        "tuned_cv": 0.9686168281485955,
        "public_lb": 0.96106,
        "public_lb_biased": 0.96895,
        "optional": False,
    },
    {
        "key": "033_blend_add032",
        "exp_id": "exp_20260609_033_blend_add032_tabm_check",
        "family": "Blend",
        "role": "add032_blend_check_output_optional",
        "oof": "oof_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy",
        "pred": "pred_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy",
        "cv": 0.9700400336552478,
        "public_lb": 0.97043,
        "optional": True,
    },
    {
        "key": "035_shared001_updated",
        "exp_id": "exp_20260610_035_shared001_updated_realmlp_pytorch_as_is",
        "family": "RealMLP",
        "role": "updated_shared001_realmlp_aux_material",
        "oof": "oof_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy",
        "pred": "pred_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy",
        "cv": 0.9687410900866934,
        "public_lb": 0.97012,
        "optional": False,
    },
    {
        "key": "037_tabicl_v2",
        "exp_id": "exp_20260610_037_tabicl_v2_as_is",
        "family": "TabICL",
        "role": "weak_tabicl_family_material_optional_corr",
        "oof": "oof_exp_20260610_037_tabicl_v2_as_is_proba.npy",
        "pred": "pred_exp_20260610_037_tabicl_v2_as_is_proba.npy",
        "cv": 0.9580770153777599,
        "public_lb": 0.95920,
        "optional": False,
    },
    {
        "key": "039_cat_v3_seed2026_qbin_variant",
        "exp_id": "exp_20260611_039_cat_v3_seed2026_qbin_variant",
        "family": "CatBoost",
        "role": "catboost_v3_lower_correlation_variant_material",
        "oof": "oof_exp_20260611_039_cat_v3_seed2026_qbin_variant_proba.npy",
        "pred": "pred_exp_20260611_039_cat_v3_seed2026_qbin_variant_proba.npy",
        "cv": 0.9687053023092482,
        "public_lb": 0.96984,
        "optional": False,
    },
    {
        "key": "044_shared001_updated_seed2027_foldvariant",
        "exp_id": "exp_20260612_044_shared001_updated_seed2027_foldvariant",
        "family": "RealMLP",
        "role": "shared001_updated_seed_fold_variant_material",
        "oof": "oof_exp_20260612_044_shared001_updated_seed2027_foldvariant_proba.npy",
        "pred": "pred_exp_20260612_044_shared001_updated_seed2027_foldvariant_proba.npy",
        "cv": 0.9686035095582338,
        "public_lb": 0.97000,
        "optional": False,
    },
    {
        "key": "048_ovr_xgb_seed2027_fe_foldvariant",
        "exp_id": "exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant",
        "family": "XGBoost",
        "role": "improved_xgb_ovr_variant_material",
        "oof": "oof_exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant_proba.npy",
        "pred": "pred_exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant_proba.npy",
        "cv": 0.9636973807714805,
        "public_lb": 0.96444,
        "optional": False,
    },
]

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("EXP_ID:", EXP_ID)
print("STACKER_SEEDS:", STACKER_SEEDS)



Using device: cuda
EXP_ID: exp_20260614_049_gpu_logreg_stacker_add048_own
STACKER_SEEDS: [42, 43, 44, 45, 46]


In [2]:

# ============================================================
# 1. Utilities
# ============================================================

class PyTorchMultiLogReg(nn.Module):
    def __init__(self, in_features, out_features=3):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def resolve_npy(filename: str, optional: bool = False):
    p = NPY_DIR / filename
    if p.exists():
        return p
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] multiple matches for {filename}. Using first:")
        for m in matches[:10]:
            print("  ", m)
        return matches[0]
    if optional:
        return None
    raise FileNotFoundError(f"Missing npy file: {filename}. Checked {NPY_DIR} and scanned {INPUT_ROOT}.")


def normalize_rows(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, EPS, None)
    p = p / p.sum(axis=1, keepdims=True)
    return p.astype(np.float32)


def validate_proba(name: str, arr: np.ndarray, expected_rows: int, n_classes: int = 3):
    assert isinstance(arr, np.ndarray), name
    assert arr.shape == (expected_rows, n_classes), f"{name}: shape {arr.shape} != {(expected_rows, n_classes)}"
    assert np.isfinite(arr).all(), f"{name}: contains non-finite values"
    row_sum = arr.sum(axis=1)
    if not np.allclose(row_sum, 1.0, atol=1e-3):
        print(f"[WARN] {name}: row sums not close to 1. Normalizing. min={row_sum.min()}, max={row_sum.max()}")
    return normalize_rows(arr)


def prob_to_logit(p):
    p = normalize_rows(p).astype(np.float64)
    p = np.clip(p, EPS, 1.0 - EPS)
    return np.clip(np.log(p / (1.0 - p)), -LOGIT_CLIP, LOGIT_CLIP).astype(np.float32)


def proba_to_pred(p):
    return np.argmax(p, axis=1)


def class_recalls(y_true, y_pred):
    out = {}
    for i, cls in INV_MAP.items():
        mask = y_true == i
        out[f"recall_{cls}"] = float((y_pred[mask] == i).mean())
    return out


def evaluate_proba(y_true, p):
    pred = proba_to_pred(p)
    out = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    out.update(class_recalls(y_true, pred))
    return out


def apply_bias_to_proba(p, bias):
    """Add class-wise log-bias to probabilities and renormalize."""
    p = normalize_rows(p).astype(np.float64)
    logp = np.log(np.clip(p, EPS, 1.0)) + np.asarray(bias, dtype=np.float64).reshape(1, -1)
    logp = logp - logp.max(axis=1, keepdims=True)
    exp = np.exp(logp)
    exp = exp / exp.sum(axis=1, keepdims=True)
    return exp.astype(np.float32)


def search_bias_coordinate(y_true, p, init_bias=None, rounds=3):
    """Simple coordinate search for class log-bias. STAR bias is allowed to move too,
    then the final vector is centered by subtracting STAR bias for readability.
    """
    if init_bias is None:
        bias = np.zeros(3, dtype=np.float64)
    else:
        bias = np.asarray(init_bias, dtype=np.float64).copy()
    best_p = apply_bias_to_proba(p, bias)
    best_score = balanced_accuracy_score(y_true, proba_to_pred(best_p))

    # coarse-to-fine coordinate grids
    grids = [
        np.arange(-1.5, 1.5001, 0.05),
        np.arange(-0.25, 0.2501, 0.01),
        np.arange(-0.06, 0.0601, 0.003),
    ]
    for r in range(rounds):
        grid = grids[min(r, len(grids) - 1)]
        improved = False
        for c in range(3):
            local_best_score = best_score
            local_best_val = bias[c]
            base_val = bias[c]
            for delta in grid:
                trial = bias.copy()
                trial[c] = base_val + float(delta)
                trial_p = apply_bias_to_proba(p, trial)
                score = balanced_accuracy_score(y_true, proba_to_pred(trial_p))
                if score > local_best_score:
                    local_best_score = score
                    local_best_val = trial[c]
            if local_best_score > best_score:
                bias[c] = local_best_val
                best_score = local_best_score
                improved = True
        print(f"bias round {r+1}: score={best_score:.9f}, bias={bias}")
        if not improved:
            break

    # Center so STAR bias is 0 for easier comparison with previous bias-search logs.
    bias = bias - bias[2]
    final_p = apply_bias_to_proba(p, bias)
    final_score = balanced_accuracy_score(y_true, proba_to_pred(final_p))
    return bias.astype(float).tolist(), float(final_score), final_p


def make_submission(sample, proba, path: Path):
    sub = sample.copy()
    pred_idx = np.argmax(proba, axis=1)
    sub[TARGET_COL] = [INV_MAP[int(i)] for i in pred_idx]
    sub.to_csv(path, index=False)
    return sub


In [3]:

# ============================================================
# 2. Load competition data and own OOF/pred assets
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

y = train[TARGET_COL].map(TARGET_MAP).astype(int).values
N = len(train)
M = len(test)
print(f"Train: {N:,} | Test: {M:,}")
print("Class distribution:", dict(zip(*np.unique(y, return_counts=True))))

loaded_oofs = []
loaded_preds = []
loaded_specs = []
skipped_specs = []

base_score_rows = []

print("\nLoading own OOF/pred assets:")
for spec in MODEL_SPECS:
    optional = bool(spec.get("optional", False))
    if spec["key"] == "033_blend_add032" and not INCLUDE_OPTIONAL_033:
        skipped_specs.append({**spec, "skip_reason": "INCLUDE_OPTIONAL_033=False"})
        continue
    try:
        oof_path = resolve_npy(spec["oof"], optional=optional)
        pred_path = resolve_npy(spec["pred"], optional=optional)
        if oof_path is None or pred_path is None:
            skipped_specs.append({**spec, "skip_reason": "optional file missing"})
            print(f"  SKIP optional {spec['key']}: file missing")
            continue
        o = np.load(oof_path)
        t = np.load(pred_path)
        o = validate_proba(f"oof:{spec['key']}", o, N, 3)
        t = validate_proba(f"pred:{spec['key']}", t, M, 3)
        loaded_oofs.append(o)
        loaded_preds.append(t)
        loaded_spec = {**spec, "resolved_oof_path": str(oof_path), "resolved_pred_path": str(pred_path)}
        loaded_specs.append(loaded_spec)

        ev = evaluate_proba(y, o)
        base_score_rows.append({
            "key": spec["key"],
            "exp_id": spec["exp_id"],
            "family": spec["family"],
            "role": spec["role"],
            "cv_from_memo": spec.get("cv", np.nan),
            "public_lb": spec.get("public_lb", np.nan),
            "public_lb_biased": spec.get("public_lb_biased", np.nan),
            "recomputed_oof_cv": ev["balanced_accuracy"],
            **{k: v for k, v in ev.items() if k.startswith("recall_")},
        })
        print(f"  {spec['key']:24s} OOF={o.shape} PRED={t.shape} CV={ev['balanced_accuracy']:.9f}")
    except Exception as e:
        if optional:
            skipped_specs.append({**spec, "skip_reason": str(e)})
            print(f"  SKIP optional {spec['key']}: {e}")
        else:
            raise

if len(loaded_oofs) == 0:
    raise RuntimeError("No own OOF/pred assets loaded.")

n_models = len(loaded_oofs)
model_keys = [s["key"] for s in loaded_specs]
print(f"\nLoaded models: {n_models}")
print(model_keys)

base_scores = pd.DataFrame(base_score_rows).sort_values("recomputed_oof_cv", ascending=False).reset_index(drop=True)
display(base_scores)
base_scores.to_csv(BASE_SCORES_PATH, index=False)
pd.DataFrame(loaded_specs).to_csv(LOADED_SPECS_PATH, index=False)
if skipped_specs:
    pd.DataFrame(skipped_specs).to_csv(OUTDIR / f"skipped_model_specs_{EXP_ID}.csv", index=False)

# Build meta features.
if USE_LOGIT_INPUT:
    meta_oofs = [prob_to_logit(o) for o in loaded_oofs]
    meta_preds = [prob_to_logit(p) for p in loaded_preds]
    input_transform = "prob_to_logit"
else:
    meta_oofs = [o.astype(np.float32) for o in loaded_oofs]
    meta_preds = [p.astype(np.float32) for p in loaded_preds]
    input_transform = "raw_probability"

X_test = np.concatenate(meta_preds, axis=1).astype(np.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
input_feature_names = [f"{key}__{cls}" for key in model_keys for cls in CLASS_NAMES]
print(f"Meta feature matrix: N x {len(input_feature_names)}")


Train: 577,347 | Test: 247,435
Class distribution: {np.int64(0): np.int64(377480), np.int64(1): np.int64(117143), np.int64(2): np.int64(82724)}

Loading own OOF/pred assets:
  015_realmlp_seed42       OOF=(577347, 3) PRED=(247435, 3) CV=0.968169345
  017_realmlp_bias         OOF=(577347, 3) PRED=(247435, 3) CV=0.968302265
  018_xgb_shared003        OOF=(577347, 3) PRED=(247435, 3) CV=0.965207986
  019_blend_best           OOF=(577347, 3) PRED=(247435, 3) CV=0.968872315
  020_blend_bias           OOF=(577347, 3) PRED=(247435, 3) CV=0.969224044
  024_seed_ensemble_blend  OOF=(577347, 3) PRED=(247435, 3) CV=0.969480311
  026_realmlp_v5           OOF=(577347, 3) PRED=(247435, 3) CV=0.969038978
  027_blend_add026         OOF=(577347, 3) PRED=(247435, 3) CV=0.969542546
  028_cat_v3               OOF=(577347, 3) PRED=(247435, 3) CV=0.968814647
  029_blend_add028         OOF=(577347, 3) PRED=(247435, 3) CV=0.970002303
  030_ovr_xgb              OOF=(577347, 3) PRED=(247435, 3) CV=0.960997130
 

,key,exp_id,family,role,cv_from_memo,public_lb,public_lb_biased,recomputed_oof_cv,recall_GALAXY,recall_QSO,recall_STAR
0,033_blend_add032,exp_20260609_033_blend_add032_tabm_check,Blend,add032_blend_check_output_optional,0.970040,0.970430,NaN,0.970040,0.961775,0.975773,0.972571
1,031_blend_add030,exp_20260608_031_blend_add030_ovr_xgb_check,Blend,public_confirmed_current_best_before_034,0.970033,0.970430,NaN,0.970033,0.961738,0.975790,0.972571
2,029_blend_add028,exp_20260608_029_blend_add028_cat_v3_check,Blend,previous_public_confirmed_and_cv_trusted_best,0.970002,0.970036,NaN,0.970002,0.961315,0.975867,0.972825
3,027_blend_add026,exp_20260608_027_blend_add026_realmlp_v5_check,Blend,previous_cv_trusted_slot,0.969543,0.969750,NaN,0.969543,0.961479,0.976439,0.970710
4,024_seed_ensemble_blend,exp_20260608_024_seed_ensemble_and_blend_check,Blend,seed_ensemble_blend_reference,0.969480,0.969580,NaN,0.969480,0.961956,0.976174,0.970311
5,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack_reference,0.969224,0.969690,NaN,0.969224,0.961137,0.976200,0.970335
6,026_realmlp_v5,exp_20260608_026_realmlp_v5_as_is,RealMLP,realmlp_v5_single_model_candidate,0.969039,0.969790,NaN,0.969039,0.959505,0.976431,0.971181
7,019_blend_best,exp_20260607_019_blend_add018_xgb_check,Blend,previous_public_confirmed_best,0.968872,0.970000,NaN,0.968872,0.965479,0.974441,0.966696
8,028_cat_v3,exp_20260608_028_cat_v3_as_is,CatBoost,catboost_v3_family_aux_material,0.968815,0.969690,NaN,0.968815,0.960099,0.974826,0.971520
9,035_shared001_updated,exp_20260610_035_shared001_updated_realmlp_pyt...,RealMLP,updated_shared001_realmlp_aux_material,0.968741,0.970120,NaN,0.968741,0.960591,0.976046,0.969586


Meta feature matrix: N x 57


In [4]:

# ============================================================
# 3. Train GPU multinomial logistic regression stacker
# ============================================================

oof_sum = np.zeros((N, 3), dtype=np.float64)
pred_sum = np.zeros((M, 3), dtype=np.float64)
fold_rows = []
seed_rows = []
last_seed_oof_feature_matrix = np.zeros((N, n_models * 3), dtype=np.float32)
last_seed_weight_rows = []

for si, seed in enumerate(STACKER_SEEDS):
    set_seed(seed)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_this = np.zeros((N, 3), dtype=np.float32)
    pred_this = np.zeros((M, 3), dtype=np.float64)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(N), y), start=1):
        t0 = time.time()
        X_tr = np.concatenate([x[tr_idx] for x in meta_oofs], axis=1).astype(np.float32)
        X_va = np.concatenate([x[va_idx] for x in meta_oofs], axis=1).astype(np.float32)
        if si == len(STACKER_SEEDS) - 1:
            last_seed_oof_feature_matrix[va_idx] = X_va

        y_tr = y[tr_idx]
        y_va = y[va_idx]

        classes = np.unique(y_tr)
        cw = compute_class_weight("balanced", classes=classes, y=y_tr)
        class_weight = dict(zip(classes, cw))
        class_weight[2] = class_weight[2] * BOOST_STAR_WEIGHT
        sample_weights = np.array([class_weight[c] for c in y_tr], dtype=np.float32)

        X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
        y_tr_t = torch.tensor(y_tr, dtype=torch.long, device=device)
        sw_t = torch.tensor(sample_weights, dtype=torch.float32, device=device)
        X_va_t = torch.tensor(X_va, dtype=torch.float32, device=device)

        model = PyTorchMultiLogReg(in_features=X_tr.shape[1], out_features=3).to(device)
        criterion = nn.CrossEntropyLoss(reduction="none")
        weight_decay_mapped = 1.0 / (C_REG * len(y_tr))
        optimizer = optim.Adam([
            {"params": model.linear.weight, "weight_decay": weight_decay_mapped},
            {"params": model.linear.bias, "weight_decay": 0.0},
        ], lr=LR)

        model.train()
        for _ in range(EPOCHS):
            optimizer.zero_grad()
            logits = model(X_tr_t)
            loss = (criterion(logits, y_tr_t) * sw_t).mean()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            va_prob = torch.softmax(model(X_va_t), dim=1).cpu().numpy().astype(np.float32)
            te_prob = torch.softmax(model(X_test_tensor), dim=1).cpu().numpy().astype(np.float64)
            weights_np = model.linear.weight.detach().cpu().numpy().astype(np.float64)
            bias_np = model.linear.bias.detach().cpu().numpy().astype(np.float64)

        oof_this[va_idx] = va_prob
        pred_this += te_prob / N_FOLDS

        y_va_pred = np.argmax(va_prob, axis=1)
        fold_score = balanced_accuracy_score(y_va, y_va_pred)
        rec = class_recalls(y_va, y_va_pred)
        row = {
            "seed": seed,
            "fold": fold,
            "balanced_accuracy": float(fold_score),
            "n_train": int(len(tr_idx)),
            "n_valid": int(len(va_idx)),
            "elapsed_sec": float(time.time() - t0),
            **rec,
        }
        fold_rows.append(row)
        print(f"Seed {seed} fold {fold}: BAC={fold_score:.9f} | {row['elapsed_sec']:.1f}s")

        # Importance from this fold's linear weights.
        for mi, key in enumerate(model_keys):
            block = weights_np[:, mi*3:(mi+1)*3]
            class_abs = np.abs(block).sum(axis=1)
            last_seed_weight_rows.append({
                "seed": seed,
                "fold": fold,
                "model_key": key,
                "importance_total": float(class_abs.sum()),
                "importance_GALAXY": float(class_abs[0]),
                "importance_QSO": float(class_abs[1]),
                "importance_STAR": float(class_abs[2]),
            })

        del model, X_tr_t, y_tr_t, sw_t, X_va_t, X_tr, X_va
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    seed_score = balanced_accuracy_score(y, np.argmax(oof_this, axis=1))
    seed_rec = class_recalls(y, np.argmax(oof_this, axis=1))
    seed_rows.append({"seed": seed, "balanced_accuracy": float(seed_score), **seed_rec})
    print(f"Seed {seed}: OOF BAC={seed_score:.9f}")

    oof_sum += oof_this
    pred_sum += pred_this

raw_oof = normalize_rows(oof_sum / len(STACKER_SEEDS))
raw_pred = normalize_rows(pred_sum / len(STACKER_SEEDS))
raw_eval = evaluate_proba(y, raw_oof)

fold_scores = pd.DataFrame(fold_rows)
seed_scores = pd.DataFrame(seed_rows)
importance_df = pd.DataFrame(last_seed_weight_rows)
importance_summary = importance_df.groupby("model_key", as_index=False).agg({
    "importance_total": "mean",
    "importance_GALAXY": "mean",
    "importance_QSO": "mean",
    "importance_STAR": "mean",
}).sort_values("importance_total", ascending=False).reset_index(drop=True)

print("\nRaw stacker evaluation:")
print(json.dumps(raw_eval, indent=2, ensure_ascii=False))
display(fold_scores.groupby("seed")["balanced_accuracy"].agg(["mean", "std"]))
display(importance_summary)

fold_scores.to_csv(FOLD_SCORES_PATH, index=False)
seed_scores.to_csv(SEED_SCORES_PATH, index=False)
importance_summary.to_csv(FEATURE_IMPORTANCE_PATH, index=False)


Seed 42 fold 1: BAC=0.970772470 | 7.9s
Seed 42 fold 2: BAC=0.969883687 | 2.5s
Seed 42 fold 3: BAC=0.969718242 | 2.5s
Seed 42 fold 4: BAC=0.969586904 | 2.6s
Seed 42 fold 5: BAC=0.970044842 | 2.6s
Seed 42: OOF BAC=0.970001228
Seed 43 fold 1: BAC=0.969787719 | 2.6s
Seed 43 fold 2: BAC=0.970836343 | 2.7s
Seed 43 fold 3: BAC=0.970887551 | 2.7s
Seed 43 fold 4: BAC=0.969680438 | 2.7s
Seed 43 fold 5: BAC=0.969517368 | 2.7s
Seed 43: OOF BAC=0.970141886
Seed 44 fold 1: BAC=0.970360544 | 2.6s
Seed 44 fold 2: BAC=0.970176940 | 2.7s
Seed 44 fold 3: BAC=0.969864332 | 2.7s
Seed 44 fold 4: BAC=0.969882137 | 2.7s
Seed 44 fold 5: BAC=0.970124801 | 2.7s
Seed 44: OOF BAC=0.970081743
Seed 45 fold 1: BAC=0.970216910 | 2.8s
Seed 45 fold 2: BAC=0.969829134 | 2.8s
Seed 45 fold 3: BAC=0.970026189 | 2.8s
Seed 45 fold 4: BAC=0.970749042 | 2.8s
Seed 45 fold 5: BAC=0.969796992 | 2.9s
Seed 45: OOF BAC=0.970123658
Seed 46 fold 1: BAC=0.970309915 | 2.9s
Seed 46 fold 2: BAC=0.970740234 | 2.9s
Seed 46 fold 3: BAC=0.9700

,mean,std
seed,,
42,0.970001,0.000464
43,0.970142,0.000665
44,0.970082,0.000210
45,0.970124,0.000388
46,0.970143,0.000452


,model_key,importance_total,importance_GALAXY,importance_QSO,importance_STAR
0,019_blend_best,0.614533,0.222939,0.184573,0.207021
1,028_cat_v3,0.607510,0.185162,0.205386,0.216962
2,032_ovr_tabm,0.600594,0.183897,0.214277,0.202420
3,048_ovr_xgb_seed2027_fe_foldvariant,0.594989,0.157242,0.216008,0.221739
4,031_blend_add030,0.594082,0.197038,0.206316,0.190728
5,026_realmlp_v5,0.584585,0.201829,0.178854,0.203902
6,024_seed_ensemble_blend,0.581471,0.169568,0.211731,0.200171
7,033_blend_add032,0.580275,0.180378,0.196457,0.203440
8,039_cat_v3_seed2026_qbin_variant,0.555885,0.205610,0.166834,0.183441
9,027_blend_add026,0.546314,0.165551,0.192427,0.188336


In [5]:

# ============================================================
# 4. Bias search and submissions
# ============================================================

best_bias, tuned_score, tuned_oof = search_bias_coordinate(y, raw_oof, init_bias=[0.0, 0.0, 0.0], rounds=3)
tuned_pred = apply_bias_to_proba(raw_pred, best_bias)
tuned_eval = evaluate_proba(y, tuned_oof)

if tuned_eval["balanced_accuracy"] >= raw_eval["balanced_accuracy"]:
    best_kind = "tuned"
    best_oof = tuned_oof
    best_pred = tuned_pred
    best_eval = tuned_eval
else:
    best_kind = "raw"
    best_oof = raw_oof
    best_pred = raw_pred
    best_eval = raw_eval

print("Raw eval:")
print(json.dumps(raw_eval, indent=2, ensure_ascii=False))
print("Tuned eval:")
print(json.dumps(tuned_eval, indent=2, ensure_ascii=False))
print("Best kind:", best_kind)
print("Best bias:", best_bias)

# Save probabilities. Default oof/pred are best_kind for future blend/corr checks.
np.save(OOF_RAW_NPY, raw_oof)
np.save(PRED_RAW_NPY, raw_pred)
np.save(OOF_TUNED_NPY, tuned_oof)
np.save(PRED_TUNED_NPY, tuned_pred)
np.save(OOF_NPY, best_oof)
np.save(PRED_NPY, best_pred)

# Save CSV versions for inspection.
pd.DataFrame(raw_oof, columns=[f"oof_raw_{c}" for c in CLASS_NAMES]).to_csv(OUTDIR / f"oof_raw_{EXP_ID}.csv", index=False)
pd.DataFrame(raw_pred, columns=[f"pred_raw_{c}" for c in CLASS_NAMES]).to_csv(OUTDIR / f"pred_raw_{EXP_ID}.csv", index=False)
pd.DataFrame(tuned_oof, columns=[f"oof_tuned_{c}" for c in CLASS_NAMES]).to_csv(OUTDIR / f"oof_tuned_{EXP_ID}.csv", index=False)
pd.DataFrame(tuned_pred, columns=[f"pred_tuned_{c}" for c in CLASS_NAMES]).to_csv(OUTDIR / f"pred_tuned_{EXP_ID}.csv", index=False)

sub_raw = make_submission(sample, raw_pred, SUBMISSION_RAW_PATH)
sub_tuned = make_submission(sample, tuned_pred, SUBMISSION_TUNED_PATH)
sub_best = make_submission(sample, best_pred, SUBMISSION_PATH)

print("Saved default OOF:", OOF_NPY.name, best_oof.shape)
print("Saved default PRED:", PRED_NPY.name, best_pred.shape)
print("Saved submission:", SUBMISSION_PATH.name)
print("Raw submission class distribution:")
print(sub_raw[TARGET_COL].value_counts(normalize=False))
print("Tuned submission class distribution:")
print(sub_tuned[TARGET_COL].value_counts(normalize=False))

cm_raw = confusion_matrix(y, np.argmax(raw_oof, axis=1))
cm_tuned = confusion_matrix(y, np.argmax(tuned_oof, axis=1))
pd.DataFrame(cm_raw, index=[f"true_{c}" for c in CLASS_NAMES], columns=[f"pred_{c}" for c in CLASS_NAMES]).to_csv(CONFUSION_RAW_PATH)
pd.DataFrame(cm_tuned, index=[f"true_{c}" for c in CLASS_NAMES], columns=[f"pred_{c}" for c in CLASS_NAMES]).to_csv(CONFUSION_TUNED_PATH)


bias round 1: score=0.970136552, bias=[-0.05  0.05  0.  ]
bias round 2: score=0.970191359, bias=[0.01 0.06 0.  ]
bias round 3: score=0.970195873, bias=[0.01  0.066 0.   ]
Raw eval:
{
  "balanced_accuracy": 0.97011955078962,
  "recall_GALAXY": 0.9604243933453428,
  "recall_QSO": 0.9766012480472585,
  "recall_STAR": 0.9733330109762584
}
Tuned eval:
{
  "balanced_accuracy": 0.9701958734042008,
  "recall_GALAXY": 0.9602733919677864,
  "recall_QSO": 0.9771988082941362,
  "recall_STAR": 0.9731154199506794
}
Best kind: tuned
Best bias: [0.010000000000001563, 0.06600000000000167, 0.0]
Saved default OOF: oof_exp_20260614_049_gpu_logreg_stacker_add048_own_proba.npy (577347, 3)
Saved default PRED: pred_exp_20260614_049_gpu_logreg_stacker_add048_own_proba.npy (247435, 3)
Saved submission: submission_exp_20260614_049_gpu_logreg_stacker_add048_own.csv
Raw submission class distribution:
class
GALAXY    156939
QSO        51402
STAR       39094
Name: count, dtype: int64
Tuned submission class distribut

In [6]:

# ============================================================
# 5. Save cv_result / registry / memo
# ============================================================

fold_std = float(fold_scores["balanced_accuracy"].std(ddof=0))
seed_std = float(seed_scores["balanced_accuracy"].std(ddof=0))

cv_result = {
    "exp_id": EXP_ID,
    "competition": COMPETITION,
    "created_at": CREATED_AT,
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "model_family": "GPU_LogisticRegression_Stacker",
    "model_type": "pytorch_multinomial_logistic_regression_on_own_oof_pred_assets",
    "raw_cv": raw_eval["balanced_accuracy"],
    "tuned_cv": tuned_eval["balanced_accuracy"],
    "best_cv": best_eval["balanced_accuracy"],
    "best_kind": best_kind,
    "best_bias": best_bias,
    "raw_recalls": {k.replace("recall_", ""): v for k, v in raw_eval.items() if k.startswith("recall_")},
    "tuned_recalls": {k.replace("recall_", ""): v for k, v in tuned_eval.items() if k.startswith("recall_")},
    "best_recalls": {k.replace("recall_", ""): v for k, v in best_eval.items() if k.startswith("recall_")},
    "n_splits": N_FOLDS,
    "n_seeds": N_SEEDS,
    "stacker_seeds": STACKER_SEEDS,
    "seed": SEED,
    "epochs": EPOCHS,
    "lr": LR,
    "C_REG": C_REG,
    "boost_star_weight": BOOST_STAR_WEIGHT,
    "use_logit_input": USE_LOGIT_INPUT,
    "input_transform": input_transform,
    "use_external_stacking_files": False,
    "n_models_loaded": n_models,
    "model_keys": model_keys,
    "loaded_specs": loaded_specs,
    "skipped_specs": skipped_specs,
    "fold_std": fold_std,
    "seed_std": seed_std,
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_raw_proba": OOF_RAW_NPY.name,
        "pred_raw_proba": PRED_RAW_NPY.name,
        "oof_tuned_proba": OOF_TUNED_NPY.name,
        "pred_tuned_proba": PRED_TUNED_NPY.name,
        "submission": SUBMISSION_PATH.name,
        "submission_raw": SUBMISSION_RAW_PATH.name,
        "submission_tuned": SUBMISSION_TUNED_PATH.name,
        "cv_result": CV_RESULT_PATH.name,
        "summary": SUMMARY_PATH.name,
        "fold_scores": FOLD_SCORES_PATH.name,
        "seed_scores": SEED_SCORES_PATH.name,
        "base_model_scores": BASE_SCORES_PATH.name,
        "loaded_model_specs": LOADED_SPECS_PATH.name,
        "feature_importance": FEATURE_IMPORTANCE_PATH.name,
    },
}
save_json(cv_result, CV_RESULT_PATH)

summary = {
    **cv_result,
    "base_scores": base_scores.to_dict(orient="records"),
    "fold_scores": fold_scores.to_dict(orient="records"),
    "seed_scores": seed_scores.to_dict(orient="records"),
    "importance_summary": importance_summary.to_dict(orient="records"),
    "submission_distribution_raw": sub_raw[TARGET_COL].value_counts().to_dict(),
    "submission_distribution_tuned": sub_tuned[TARGET_COL].value_counts().to_dict(),
    "reference_scores": {
        "033_cv": 0.9700400336552478,
        "033_public_lb": 0.97043,
        "036_gpu_logreg_add035_cv": 0.9700674063284508,
        "036_gpu_logreg_add035_public_lb": 0.97037,
        "036_blend_add035_cv": 0.9700727843277738,
        "036_blend_add035_public_lb": 0.97023,
        "031_cv": 0.9700333620193362,
        "031_public_lb": 0.97043,
        "037_cv": 0.9580770153777599,
        "037_public_lb": 0.95920,
        "039_cv": 0.9687053023092482,
        "039_public_lb": 0.96984,
        "040_gpu_logreg_add039_cv": 0.9702017877238539,
        "040_gpu_logreg_add039_public_lb": 0.97069,
        "043_gpu_logreg_add042_cv": 0.9701820598346598,
        "043_gpu_logreg_add042_public_lb": 0.97034,
        "044_cv": 0.9686035095582338,
        "044_public_lb": 0.97000,
        "045_gpu_logreg_add044_cv": 0.9702445493383917,
        "045_gpu_logreg_add044_public_lb": 0.97040,
        "046_blend_add042_044_045_cv": 0.9702533348410616,
        "046_blend_add042_044_045_public_lb": 0.97042,
        "047_xgb_multiclass_cv": 0.9596874103404002,
        "047_xgb_multiclass_public_lb": 0.96027,
        "048_ovr_xgb_seed2027_fe_foldvariant_cv": 0.9636973807714805,
        "048_ovr_xgb_seed2027_fe_foldvariant_public_lb": 0.96444,
        "038_gpu_logreg_add037_cv": 0.9701353365798572,
        "038_gpu_logreg_add037_public_lb": 0.97060,
        "029_cv": 0.9700023026377228,
        "029_public_lb": 0.970036,
        "032_raw_cv": 0.9610105651284856,
        "032_tuned_cv": 0.9686168281485955,
        "032_raw_public_lb": 0.96106,
        "032_biased_public_lb": 0.96895,
    },
}
save_json(summary, SUMMARY_PATH)

# Registry update
registry_path = WORKDIR / "oof_registry.csv"
registry_row = {
    "exp_id": EXP_ID,
    "model_family": "GPU_LogisticRegression_Stacker",
    "feature_family": "own_oof_pred_assets_logit_meta_features",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(best_eval["balanced_accuracy"]),
    "fold_std": fold_std,
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": False,
    "use_bias_search": True,
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
    "submission_path": str(SUBMISSION_PATH),
    "role": "self_owned_gpu_logreg_stacker_candidate",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb_and_corr",
    "notes": (
        "GPU PyTorch multinomial logistic regression stacker add048 using only self-owned OOF/pred npy assets. "
        "External stacking/submission files are not used. Default OOF/pred are best of raw vs bias-tuned stacker proba by OOF CV."
    ),
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "GPU logistic regression stacker add048 using own OOF/pred assets including 048 OVR XGB variant",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "PyTorchMultiLogReg",
        "created_at": CREATED_AT,
    },
    "objective": {
        "summary": (
            "Train a GPU multinomial logistic regression stacker using only self-owned OOF/pred assets "
            "and save exp_id-named OOF/pred probabilities for submission and later diagnostics."
        ),
        "success_criteria": [
            "load only self-owned OOF/pred npy files",
            "exclude external submission and public notebook predictions",
            "train fold-safe GPU logistic regression stacker",
            "save raw and tuned OOF/pred probabilities",
            "save default best OOF/pred probabilities with exp_id in filename",
            "save raw/tuned/default submissions",
            "save cv_result, fold_scores, seed_scores, base_model_scores, importance",
            "update oof_registry",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "target_col": TARGET_COL,
        "id_col": ID_COL,
        "use_original": False,
        "use_external_stacking_files": False,
        "n_models_loaded": n_models,
        "loaded_model_keys": model_keys,
        "skipped_specs": skipped_specs,
    },
    "seed": {
        "seed": SEED,
        "stacker_seeds": STACKER_SEEDS,
        "stratified_kfold_random_states": STACKER_SEEDS,
        "numpy_random_seed": "set per stacker seed",
        "torch_random_seed": "set per stacker seed",
    },
    "features": {
        "feature_family": "own_oof_pred_assets_logit_meta_features",
        "input_transform": input_transform,
        "use_logit_input": USE_LOGIT_INPUT,
        "n_meta_features": len(input_feature_names),
        "input_feature_names": input_feature_names,
        "note": (
            "Each base model contributes 3 probability-derived features. "
            "With USE_LOGIT_INPUT=True, class probabilities are clipped and transformed by log(p/(1-p))."
        ),
    },
    "cv": {
        "scheme": "StratifiedKFold meta-stacker CV",
        "n_splits": N_FOLDS,
        "n_seeds": N_SEEDS,
        "shuffle": True,
        "raw_score": float(raw_eval["balanced_accuracy"]),
        "tuned_score": float(tuned_eval["balanced_accuracy"]),
        "best_score": float(best_eval["balanced_accuracy"]),
        "best_kind": best_kind,
        "best_bias": best_bias,
        "fold_scores": fold_scores.to_dict(orient="records"),
        "seed_scores": seed_scores.to_dict(orient="records"),
        "fold_std": fold_std,
        "seed_std": seed_std,
        "raw_recalls": {k.replace("recall_", ""): v for k, v in raw_eval.items() if k.startswith("recall_")},
        "tuned_recalls": {k.replace("recall_", ""): v for k, v in tuned_eval.items() if k.startswith("recall_")},
    },
    "model": {
        "model_family": "GPU_LogisticRegression_Stacker",
        "implementation": "PyTorch linear layer + CrossEntropyLoss",
        "epochs": EPOCHS,
        "lr": LR,
        "C_REG": C_REG,
        "weight_decay_mapping": "1 / (C_REG * n_train)",
        "class_weight": "balanced class weights per fold",
        "boost_star_weight": BOOST_STAR_WEIGHT,
        "device": str(device),
    },
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_raw_proba": OOF_RAW_NPY.name,
        "pred_raw_proba": PRED_RAW_NPY.name,
        "oof_tuned_proba": OOF_TUNED_NPY.name,
        "pred_tuned_proba": PRED_TUNED_NPY.name,
        "submission": SUBMISSION_PATH.name,
        "submission_raw": SUBMISSION_RAW_PATH.name,
        "submission_tuned": SUBMISSION_TUNED_PATH.name,
        "cv_result": CV_RESULT_PATH.name,
        "summary": SUMMARY_PATH.name,
        "fold_scores": FOLD_SCORES_PATH.name,
        "seed_scores": SEED_SCORES_PATH.name,
        "base_model_scores": BASE_SCORES_PATH.name,
        "loaded_model_specs": LOADED_SPECS_PATH.name,
        "feature_importance": FEATURE_IMPORTANCE_PATH.name,
        "registry": "oof_registry.csv",
        "memo": "memo.yml",
    },
    "judgement": {
        "status": "pending_public_lb",
        "reason": (
            "This is a self-owned GPU logistic regression stacker. Adoption depends on OOF CV, "
            "Public LB, and whether adding 048 improves over the current 040 GPU LogReg add039 reference and 045 add044 diagnostic."
        ),
        "current_reference": {
            "033_cv": 0.9700400336552478,
            "033_public_lb": 0.97043,
            "036_gpu_logreg_add035_cv": 0.9700674063284508,
            "036_gpu_logreg_add035_public_lb": 0.97037,
            "040_gpu_logreg_add039_cv": 0.9702017877238539,
            "040_gpu_logreg_add039_public_lb": 0.97069,
            "038_gpu_logreg_add037_cv": 0.9701353365798572,
            "038_gpu_logreg_add037_public_lb": 0.97060,
            "039_cat_v3_variant_cv": 0.9687053023092482,
            "039_cat_v3_variant_public_lb": 0.96984,
            "044_shared001_updated_variant_cv": 0.9686035095582338,
            "044_shared001_updated_variant_public_lb": 0.97000,
            "045_gpu_logreg_add044_cv": 0.9702445493383917,
            "045_gpu_logreg_add044_public_lb": 0.97040,
            "046_blend_add042_044_045_cv": 0.9702533348410616,
            "046_blend_add042_044_045_public_lb": 0.97042,
            "047_xgb_multiclass_variant_cv": 0.9596874103404002,
            "047_xgb_multiclass_variant_public_lb": 0.96027,
            "048_ovr_xgb_seed2027_fe_foldvariant_cv": 0.9636973807714805,
            "048_ovr_xgb_seed2027_fe_foldvariant_public_lb": 0.96444,
            "031_cv": 0.9700333620193362,
            "031_public_lb": 0.97043,
        },
        "risk_notes": [
            "Base blend assets such as 029/031 may themselves contain OOF-optimized HC weights.",
            "Bias search is optimized on OOF labels and should be validated by Public LB before promotion.",
            "Do not treat in-sample-looking meta improvements as final without LB confirmation.",
        ],
        "next_action": [
            "Submit submission_raw and submission_tuned if both are generated and quota allows, otherwise submit the default best submission first.",
            "Record Public LB for raw/tuned/default submission and compare against 040 GPU LogReg add039 / 045 add044 / 046 CV-best blend / 038 GPU LogReg add037 / 033.",
            "If Public LB improves 040 GPU LogReg add039, promote 049 to current best.",
            "If Public LB does not improve, keep 040 GPU LogReg add039 as slot_1 and 033 as backup/diversity slot; keep 049 as diagnostic/add048 material.",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("cv_result saved:", CV_RESULT_PATH)
print("summary saved:", SUMMARY_PATH)
print("registry saved:", registry_path)
print("memo saved:", memo_path)
display(registry.tail())


cv_result saved: /kaggle/working/exp_20260614_049_gpu_logreg_stacker_add048_own/cv_result_exp_20260614_049_gpu_logreg_stacker_add048_own.json
summary saved: /kaggle/working/exp_20260614_049_gpu_logreg_stacker_add048_own/gpu_logreg_stacker_summary.json
registry saved: /kaggle/working/oof_registry.csv
memo saved: /kaggle/working/exp_20260614_049_gpu_logreg_stacker_add048_own/memo.yml


,exp_id,model_family,feature_family,cv_metric,cv_score,fold_std,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260614_049_gpu_logreg_stacker_add048_own,GPU_LogisticRegression_Stacker,own_oof_pred_assets_logit_meta_features,balanced_accuracy_score,0.970196,0.000415,NaN,False,False,True,/kaggle/working/exp_20260614_049_gpu_logreg_st...,/kaggle/working/exp_20260614_049_gpu_logreg_st...,/kaggle/working/exp_20260614_049_gpu_logreg_st...,self_owned_gpu_logreg_stacker_candidate,completed,pending_public_lb_and_corr,GPU PyTorch multinomial logistic regression st...
